In [10]:
"""
BioHub Cell Tracking pipeline.

This file follows the project pseudocode for the current dataset layout:
time-lapse grayscale PNG images stored in sequence folders under
dataset-smaller.

For competition-style use, the preferred path is supervised segmentation:
PyTorch Dataset -> DataLoader -> U-Net -> Dice+BCE loss -> AdamW -> checkpoint
-> inference -> post-processing -> tracking.

The current dataset-smaller folder has no ground-truth masks, so training
requires a separate mask folder. Classical segmentation is still available as
an explicit fallback/debug mode.
"""

# Standard library imports

from __future__ import annotations

import argparse
import math
import os
import random
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple
from typing import Any, cast

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFilter, ImageOps


import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset



from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment
from skimage import filters, measure, morphology, segmentation



device = torch.device("cpu")

FRAME_RE = re.compile(r"t(\d+)\.png$", re.IGNORECASE)

# Central configuration used by training, inference, and tracking.


@dataclass
class Config:
    dataset_dir: Path = Path("dataset-smaller")
    output_dir: Path = Path("outputs")
    train_mask_dir: Optional[Path] = None
    checkpoint_path: Optional[Path] = None
    seed: int = 42

    image_size: int = 512
    batch_size: int = 16
    epochs: int = 50
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    val_split: float = 0.2
    num_workers: int = 8
    confidence_threshold: float = 0.5
    segmentation_method: str = "auto"

    min_area_default: int = 20
    max_area_default: int = 100000
    max_tracking_distance: float = 65.0
    max_missing_frames: int = 3
    match_cost_threshold: float = 0.75

    distance_weight: float = 0.45
    iou_weight: float = 0.30
    area_weight: float = 0.15
    intensity_weight: float = 0.10

    visualize_every: int = 25
    save_masks: bool = True
    save_visualizations: bool = True

    limit_frames: Optional[int] = None
    only_sequence: Optional[str] = None

    @property
    def masks_dir(self) -> Path:
        return self.output_dir / "masks"

    @property
    def tracks_dir(self) -> Path:
        return self.output_dir / "tracks"

    @property
    def visualizations_dir(self) -> Path:
        return self.output_dir / "visualizations"

    @property
    def logs_dir(self) -> Path:
        return self.output_dir / "logs"

    @property
    def checkpoint_dir(self) -> Path:
        return self.output_dir / "checkpoints"


@dataclass
class SequenceRecord:
    sequence_id: str
    folder: Path
    frame_paths: List[Path]
    frame_numbers: List[int]
    image_size: Tuple[int, int]
    bit_depth: int


@dataclass
class TrainingPair:
    sequence_id: str
    frame_number: int
    image_path: Path
    mask_path: Path


@dataclass
class Detection:
    sequence_id: str
    frame_number: int
    detection_id: int
    label_id: int
    area: int
    centroid_x: float
    centroid_y: float
    bbox_x1: int
    bbox_y1: int
    bbox_x2: int
    bbox_y2: int
    mean_intensity: float
    perimeter: float
    circularity: float
    mask_path: str = ""
    track_id: Optional[int] = None
    parent_id: int = 0

    @property
    def centroid(self) -> Tuple[float, float]:
        return self.centroid_x, self.centroid_y

    @property
    def bbox(self) -> Tuple[int, int, int, int]:
        return self.bbox_x1, self.bbox_y1, self.bbox_x2, self.bbox_y2


@dataclass
class Track:
    track_id: int
    parent_id: int = 0
    detections: List[Detection] = field(default_factory=list)
    missing_count: int = 0
    active: bool = True

    @property
    def last_detection(self) -> Detection:
        return self.detections[-1]

    @property
    def last_frame(self) -> int:
        return self.last_detection.frame_number

    def add_detection(self, detection: Detection) -> None:
        detection.track_id = self.track_id
        detection.parent_id = self.parent_id
        self.detections.append(detection)
        self.missing_count = 0
        self.active = True

    def predicted_centroid(self) -> Tuple[float, float]:
        if len(self.detections) < 2:
            return self.last_detection.centroid

        prev = self.detections[-2]
        curr = self.detections[-1]
        dx = curr.centroid_x - prev.centroid_x
        dy = curr.centroid_y - prev.centroid_y
        return curr.centroid_x + dx, curr.centroid_y + dy


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)


def create_output_folders(config: Config) -> None:
    for folder in [
        config.output_dir,
        config.masks_dir,
        config.tracks_dir,
        config.visualizations_dir,
        config.logs_dir,
        config.checkpoint_dir,
    ]:
        folder.mkdir(parents=True, exist_ok=True)


def frame_number_from_path(path: Path) -> int:
    match = FRAME_RE.match(path.name)
    if not match:
        raise ValueError(f"Invalid frame filename: {path.name}")
    return int(match.group(1))


def read_png_header(path: Path) -> Tuple[int, int, int, int]:
    with path.open("rb") as fh:
        signature = fh.read(8)
        if signature != b"\x89PNG\r\n\x1a\n":
            raise ValueError(f"Not a PNG file: {path}")
        length = int.from_bytes(fh.read(4), "big")
        chunk_type = fh.read(4)
        data = fh.read(length)
        if chunk_type != b"IHDR" or len(data) < 13:
            raise ValueError(f"Invalid PNG header: {path}")
        width = int.from_bytes(data[0:4], "big")
        height = int.from_bytes(data[4:8], "big")
        bit_depth = data[8]
        color_type = data[9]
        return width, height, bit_depth, color_type
    
# Scan the dataset directory, validate frame continuity,
# and collect metadata for every sequence.

def discover_sequences(config: Config) -> List[SequenceRecord]:
    dataset_dir = config.dataset_dir
    if not dataset_dir.exists():
        raise FileNotFoundError(f"Dataset folder not found: {dataset_dir}")

    sequences: List[SequenceRecord] = []
    for folder in sorted(path for path in dataset_dir.iterdir() if path.is_dir()):
        if config.only_sequence and folder.name != config.only_sequence:
            continue

        frame_paths = [p for p in folder.glob("*.png") if FRAME_RE.match(p.name)]
        frame_paths = sorted(frame_paths, key=frame_number_from_path)
        if not frame_paths:
            continue

        frame_numbers = [frame_number_from_path(p) for p in frame_paths]
        expected = list(range(frame_numbers[0], frame_numbers[-1] + 1))
        if frame_numbers != expected:
            missing = sorted(set(expected) - set(frame_numbers))
            raise ValueError(f"{folder.name} has missing frames: {missing[:10]}")

        width, height, bit_depth, color_type = read_png_header(frame_paths[0])
        for path in frame_paths[1:]:
            w, h, bd, ct = read_png_header(path)
            if (w, h, bd, ct) != (width, height, bit_depth, color_type):
                raise ValueError(f"Inconsistent image metadata in {folder.name}: {path}")

        if config.limit_frames is not None:
            frame_paths = frame_paths[: config.limit_frames]
            frame_numbers = frame_numbers[: config.limit_frames]

        sequences.append(
            SequenceRecord(
                sequence_id=folder.name,
                folder=folder,
                frame_paths=frame_paths,
                frame_numbers=frame_numbers,
                image_size=(width, height),
                bit_depth=bit_depth,
            )
        )

    return sequences

# Load a frame and normalize its intensity using percentile clipping
# to reduce illumination variation across microscopy images.

def load_frame(path: Path) -> np.ndarray:
    with Image.open(path) as image:
        array = np.asarray(image).astype(np.float32)

    if array.ndim == 3:
        array = array.mean(axis=2)

    percentiles = np.percentile(array, [1.0, 99.5])
    low, high = float(percentiles[0]), float(percentiles[1])
    if high <= low:
        high = float(array.max())
        low = float(array.min())

    if high <= low:
        return np.zeros_like(array, dtype=np.float32)

    array = np.clip(array, low, high)
    array = (array - low) / (high - low)
    return array.astype(np.float32)


def require_torch() -> None:
    if torch is None:
        raise RuntimeError(
            "PyTorch is required for training/model inference, but it is not installed "
            "in this Python environment. Install torch or use --segmentation classical."
        )


def get_torch_device() -> "torch.device":
    require_torch()
    return torch.device("cpu")


def load_binary_mask(path: Path) -> np.ndarray:
    with Image.open(path) as image:
        array = np.asarray(image)
    if array.ndim == 3:
        array = array[..., 0]
    return (array > 0).astype(np.float32)


def resize_array(array: np.ndarray, size: int, is_mask: bool = False) -> np.ndarray:
    if size <= 0:
        return array

    mode = "L"
    if is_mask:
        pil = Image.fromarray((array > 0).astype(np.uint8) * 255, mode=mode)
        resample = Image.Resampling.NEAREST
    else:
        pil = Image.fromarray(float_to_uint8(array), mode=mode)
        resample = Image.Resampling.BILINEAR

    resized = pil.resize((size, size), resample=resample)
    out = np.asarray(resized).astype(np.float32) / 255.0
    if is_mask:
        out = (out > 0.5).astype(np.float32)
    return out

# Random augmentations used only during training
# to improve segmentation generalization.


def augment_pair(image: np.ndarray, mask: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    if random.random() < 0.5:
        image = np.flip(image, axis=1)
        mask = np.flip(mask, axis=1)
    if random.random() < 0.5:
        image = np.flip(image, axis=0)
        mask = np.flip(mask, axis=0)

    rotations = random.randint(0, 3)
    if rotations:
        image = np.rot90(image, rotations)
        mask = np.rot90(mask, rotations)

    if random.random() < 0.35:
        gain = random.uniform(0.85, 1.20)
        bias = random.uniform(-0.08, 0.08)
        image = np.clip(image * gain + bias, 0.0, 1.0)

    if random.random() < 0.25:
        noise = np.random.normal(0.0, 0.025, size=image.shape).astype(np.float32)
        image = np.clip(image + noise, 0.0, 1.0)

    return np.ascontiguousarray(image), np.ascontiguousarray(mask)


def mask_candidates(mask_root: Path, sequence_id: str, frame_stem: str) -> List[Path]:
    seq_root = mask_root / sequence_id
    names = [
        f"{frame_stem}.png",
        f"{frame_stem}_mask.png",
        f"{frame_stem}.tif",
        f"{frame_stem}.tiff",
        f"{frame_stem}_mask.tif",
        f"{frame_stem}_mask.tiff",
    ]
    candidates = [seq_root / name for name in names]
    candidates.extend(mask_root / name for name in names)
    return candidates


def discover_training_pairs(sequences: Sequence[SequenceRecord], config: Config) -> List[TrainingPair]:
    if config.train_mask_dir is None:
        raise ValueError("Training requires --mask-dir pointing to ground-truth masks.")

    mask_root = config.train_mask_dir
    if not mask_root.exists():
        raise FileNotFoundError(f"Mask folder not found: {mask_root}")

    pairs: List[TrainingPair] = []
    missing = 0

    for sequence in sequences:
        for frame_path, frame_number in zip(sequence.frame_paths, sequence.frame_numbers):
            mask_path = next((p for p in mask_candidates(mask_root, sequence.sequence_id, frame_path.stem) if p.exists()), None)
            if mask_path is None:
                missing += 1
                continue
            pairs.append(
                TrainingPair(
                    sequence_id=sequence.sequence_id,
                    frame_number=frame_number,
                    image_path=frame_path,
                    mask_path=mask_path,
                )
            )

    if not pairs:
        raise RuntimeError(
            f"No image-mask pairs found. Expected masks like "
            f"{mask_root / '<sequence>' / '<frame>_mask.png'}."
        )

    if missing:
        print(f"Warning: {missing} images did not have a matching mask and were skipped.")

    return pairs

# PyTorch dataset returning image-mask pairs for supervised training.

class CellSegmentationDataset(Dataset):
    def __init__(self, pairs: Sequence[TrainingPair], image_size: int, training: bool):
        self.pairs = list(pairs)
        self.image_size = image_size
        self.training = training

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, index: int):
        require_torch()
        pair = self.pairs[index]
        image = load_frame(pair.image_path)
        mask = load_binary_mask(pair.mask_path)

        image = resize_array(image, self.image_size, is_mask=False)
        mask = resize_array(mask, self.image_size, is_mask=True)

        if self.training:
            image, mask = augment_pair(image, mask)

        image_tensor = torch.from_numpy(image[None, ...].astype(np.float32))
        mask_tensor = torch.from_numpy(mask[None, ...].astype(np.float32))
        return image_tensor, mask_tensor


if nn is not None:

    class ConvBlock(nn.Module):
        def __init__(self, in_channels: int, out_channels: int):
            super().__init__()
            self.block = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
            )

        def forward(self, x):
            return self.block(x)
        
        # Lightweight U-Net encoder-decoder architecture
        # for binary cell segmentation.


    class UNetSmall(nn.Module):
        def __init__(self, in_channels: int = 1, out_channels: int = 1, base_channels: int = 32):
            super().__init__()
            self.enc1 = ConvBlock(in_channels, base_channels)
            self.enc2 = ConvBlock(base_channels, base_channels * 2)
            self.enc3 = ConvBlock(base_channels * 2, base_channels * 4)
            self.enc4 = ConvBlock(base_channels * 4, base_channels * 8)

            self.pool = nn.MaxPool2d(2)
            self.bottleneck = ConvBlock(base_channels * 8, base_channels * 16)

            self.up4 = nn.ConvTranspose2d(base_channels * 16, base_channels * 8, kernel_size=2, stride=2)
            self.dec4 = ConvBlock(base_channels * 16, base_channels * 8)
            self.up3 = nn.ConvTranspose2d(base_channels * 8, base_channels * 4, kernel_size=2, stride=2)
            self.dec3 = ConvBlock(base_channels * 8, base_channels * 4)
            self.up2 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, kernel_size=2, stride=2)
            self.dec2 = ConvBlock(base_channels * 4, base_channels * 2)
            self.up1 = nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=2, stride=2)
            self.dec1 = ConvBlock(base_channels * 2, base_channels)

            self.head = nn.Conv2d(base_channels, out_channels, kernel_size=1)

        def forward(self, x):
            e1 = self.enc1(x)
            e2 = self.enc2(self.pool(e1))
            e3 = self.enc3(self.pool(e2))
            e4 = self.enc4(self.pool(e3))

            b = self.bottleneck(self.pool(e4))

            d4 = self.up4(b)
            d4 = pad_to_match(d4, e4)
            d4 = self.dec4(torch.cat([d4, e4], dim=1))
            d3 = self.up3(d4)
            d3 = pad_to_match(d3, e3)
            d3 = self.dec3(torch.cat([d3, e3], dim=1))
            d2 = self.up2(d3)
            d2 = pad_to_match(d2, e2)
            d2 = self.dec2(torch.cat([d2, e2], dim=1))
            d1 = self.up1(d2)
            d1 = pad_to_match(d1, e1)
            d1 = self.dec1(torch.cat([d1, e1], dim=1))
            return self.head(d1)


def pad_to_match(x, target):
    diff_y = target.size(2) - x.size(2)
    diff_x = target.size(3) - x.size(3)
    if diff_x == 0 and diff_y == 0:
        return x
    return F.pad(x, [diff_x // 2, diff_x - diff_x // 2, diff_y // 2, diff_y - diff_y // 2])


def build_model() -> "nn.Module":
    require_torch()
    if nn is None:
        raise RuntimeError("torch.nn is unavailable.")
    return UNetSmall(in_channels=1, out_channels=1, base_channels=32)


def dice_loss_from_logits(logits, targets, smooth: float = 1.0):
    probs = torch.sigmoid(logits)
    dims = (1, 2, 3)
    intersection = torch.sum(probs * targets, dims)
    union = torch.sum(probs, dims) + torch.sum(targets, dims)
    dice = (2.0 * intersection + smooth) / (union + smooth)
    return 1.0 - dice.mean()


def combined_loss(logits, targets):
    bce = F.binary_cross_entropy_with_logits(logits, targets)
    dice = dice_loss_from_logits(logits, targets)
    return bce + dice


def segmentation_metrics_from_logits(logits, targets, threshold: float = 0.5) -> Dict[str, float]:
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    targets = (targets > 0.5).float()

    tp = torch.sum(preds * targets).item()
    fp = torch.sum(preds * (1.0 - targets)).item()
    fn = torch.sum((1.0 - preds) * targets).item()

    dice = (2.0 * tp + 1.0) / (2.0 * tp + fp + fn + 1.0)
    iou = (tp + 1.0) / (tp + fp + fn + 1.0)
    precision = (tp + 1.0) / (tp + fp + 1.0)
    recall = (tp + 1.0) / (tp + fn + 1.0)
    return {"dice": dice, "iou": iou, "precision": precision, "recall": recall}


def split_training_pairs(pairs: Sequence[TrainingPair], val_split: float, seed: int) -> Tuple[List[TrainingPair], List[TrainingPair]]:
    pairs = list(pairs)
    random.Random(seed).shuffle(pairs)
    val_count = max(1, int(len(pairs) * val_split)) if len(pairs) > 1 else 0
    val_pairs = pairs[:val_count]
    train_pairs = pairs[val_count:]
    if not train_pairs:
        train_pairs, val_pairs = pairs, []
    return train_pairs, val_pairs


def train_one_epoch(model, loader, optimizer, device) -> float:
    model.train()
    total_loss = 0.0
    total_items = 0

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = combined_loss(logits, masks)
        loss.backward()
        optimizer.step()

        batch_size = images.size(0)
        total_loss += float(loss.item()) * batch_size
        total_items += batch_size

    return total_loss / max(total_items, 1)


def validate_model(model, loader, device, threshold: float) -> Dict[str, float]:
    if loader is None:
        return {"loss": 0.0, "dice": 0.0, "iou": 0.0, "precision": 0.0, "recall": 0.0}

    model.eval()
    total_loss = 0.0
    total_items = 0
    totals = {"dice": 0.0, "iou": 0.0, "precision": 0.0, "recall": 0.0}

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)
            logits = model(images)
            loss = combined_loss(logits, masks)
            metrics = segmentation_metrics_from_logits(logits, masks, threshold=threshold)

            batch_size = images.size(0)
            total_loss += float(loss.item()) * batch_size
            total_items += batch_size
            for key in totals:
                totals[key] += metrics[key] * batch_size

    result = {"loss": total_loss / max(total_items, 1)}
    result.update({key: value / max(total_items, 1) for key, value in totals.items()})
    return result

# Complete supervised training pipeline including
# data loading, optimization, validation,
# checkpointing, and metric logging.

def train_model(config: Config) -> Path:
    require_torch()
    create_output_folders(config)
    set_seed(config.seed)
    device = get_torch_device()

    sequences = discover_sequences(config)
    pairs = discover_training_pairs(sequences, config)
    train_pairs, val_pairs = split_training_pairs(pairs, config.val_split, config.seed)

    train_dataset = CellSegmentationDataset(train_pairs, image_size=config.image_size, training=True)
    val_dataset = CellSegmentationDataset(val_pairs, image_size=config.image_size, training=False) if val_pairs else None

    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=config.num_workers,
        pin_memory=False,
    )
    val_loader = (
        DataLoader(
            val_dataset,
            batch_size=config.batch_size,
            shuffle=False,
            num_workers=config.num_workers,
            pin_memory=False,
        )
        if val_dataset is not None
        else None
    )

    model = build_model().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(config.epochs, 1))

    best_score = -1.0
    best_path = config.checkpoint_dir / "best_unet.pt"
    history_rows = []

    print(f"Training on {len(train_pairs)} pairs, validating on {len(val_pairs)} pairs, device={device}")
    for epoch in range(1, config.epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, device)
        val_metrics = validate_model(model, val_loader, device, threshold=config.confidence_threshold)
        scheduler.step()

        score = val_metrics["dice"] if val_pairs else -train_loss
        lr = optimizer.param_groups[0]["lr"]
        history_rows.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_metrics["loss"],
                "dice": val_metrics["dice"],
                "iou": val_metrics["iou"],
                "precision": val_metrics["precision"],
                "recall": val_metrics["recall"],
                "learning_rate": lr,
            }
        )

        print(
            f"Epoch {epoch:03d}/{config.epochs} "
            f"train_loss={train_loss:.4f} val_dice={val_metrics['dice']:.4f} "
            f"val_iou={val_metrics['iou']:.4f} lr={lr:.6g}"
        )

        if score > best_score:
            best_score = score
            torch.save(
                {
                    "model_state": model.state_dict(),
                    "epoch": epoch,
                    "best_score": best_score,
                    "config": {
                        "image_size": config.image_size,
                        "confidence_threshold": config.confidence_threshold,
                    },
                },
                best_path,
            )

    pd.DataFrame(history_rows).to_csv(config.logs_dir / "training_history.csv", index=False)
    print(f"Best checkpoint saved to {best_path}")
    return best_path


def load_checkpoint_model(config: Config):
    require_torch()
    checkpoint_path = config.checkpoint_path or (config.checkpoint_dir / "best_unet.pt")
    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Checkpoint not found: {checkpoint_path}. Train first or pass --checkpoint."
        )

    device = get_torch_device()
    model = build_model().to(device)
    if hasattr(torch, "compile"):
     model = torch.compile(model)
     from typing import Any, cast

    checkpoint = cast(dict[str, Any], torch.load(checkpoint_path, map_location=device))
    model.load_state_dict(checkpoint["model_state"])
    model.eval()
    return model, device

# Perform segmentation using a trained U-Net model.

def segment_cells_with_model(image: np.ndarray, model, device, config: Config) -> np.ndarray:
    require_torch()
    original_h, original_w = image.shape[:2]
    model_input = resize_array(image, config.image_size, is_mask=False)
    tensor = torch.from_numpy(model_input[None, None, ...].astype(np.float32)).to(device)

    with torch.no_grad():
        logits = model(tensor)
        logits = F.interpolate(logits, size=(original_h, original_w), mode="bilinear", align_corners=False)
        probs = torch.sigmoid(logits)[0, 0].detach().cpu().numpy()

    binary = probs > config.confidence_threshold
    return split_and_label(binary, min_distance=4)


def gaussian_blur(image: np.ndarray, radius: float = 1.2) -> np.ndarray:
    pil = Image.fromarray(float_to_uint8(image), mode="L")
    blurred = pil.filter(ImageFilter.GaussianBlur(radius=radius))
    return np.asarray(blurred).astype(np.float32) / 255.0


def preprocess_frame(image: np.ndarray) -> np.ndarray:
    blurred = gaussian_blur(image, radius=0.8)
    enhanced = np.clip(image * 0.75 + blurred * 0.25, 0.0, 1.0)
    return enhanced.astype(np.float32)


def float_to_uint8(image: np.ndarray) -> np.ndarray:
    image = np.nan_to_num(image, nan=0.0, posinf=1.0, neginf=0.0)
    return (np.clip(image, 0.0, 1.0) * 255.0).round().astype(np.uint8)


def otsu_threshold(image: np.ndarray) -> float:
    if filters is not None:
        return float(filters.threshold_otsu(image))

    hist, edges = np.histogram(image.ravel(), bins=256, range=(0.0, 1.0))
    total = image.size
    sum_total = np.dot(hist, edges[:-1])
    sum_background = 0.0
    weight_background = 0.0
    best_var = -1.0
    best_threshold = 0.5

    for idx, count in enumerate(hist):
        weight_background += count
        if weight_background == 0:
            continue
        weight_foreground = total - weight_background
        if weight_foreground == 0:
            break

        sum_background += edges[idx] * count
        mean_background = sum_background / weight_background
        mean_foreground = (sum_total - sum_background) / weight_foreground
        between_var = weight_background * weight_foreground
        between_var *= (mean_background - mean_foreground) ** 2

        if between_var > best_var:
            best_var = between_var
            best_threshold = float(edges[idx])

    return best_threshold


def sequence_family(sequence_id: str) -> str:
    return sequence_id.split("-")[0].lower()

# Classical image-processing fallback used when
# no trained model is available.

def segment_cells(image: np.ndarray, sequence_id: str) -> np.ndarray:
    family = sequence_family(sequence_id)

    if family == "hela":
        smoothed = gaussian_blur(image, radius=1.0)
        threshold = max(otsu_threshold(smoothed), float(np.percentile(smoothed, 90)))
        binary = smoothed > threshold
        return split_and_label(binary, min_distance=4)

    if family == "pancreas":
        smoothed = gaussian_blur(image, radius=1.0)
        high_pass = np.clip(smoothed - gaussian_blur(smoothed, radius=5.0), 0.0, 1.0)
        threshold = max(otsu_threshold(high_pass), float(np.percentile(high_pass, 75)))
        binary = high_pass > threshold
        return split_and_label(binary, min_distance=3)

    if family in {"hsc", "musc"}:
        smoothed = gaussian_blur(image, radius=1.0)
        background = gaussian_blur(smoothed, radius=8.0)
        contrast = np.abs(smoothed - background)
        threshold = max(otsu_threshold(contrast), float(np.percentile(contrast, 92)))
        binary = contrast > threshold
        return split_and_label(binary, min_distance=6)

    threshold = otsu_threshold(image)
    return split_and_label(image > threshold, min_distance=4)

# Separate touching cells using distance transform
# followed by watershed segmentation.

def split_and_label(binary: np.ndarray, min_distance: int) -> np.ndarray:
    binary = binary.astype(bool)
    binary = binary_open(binary, radius=1)
    binary = binary_close(binary, radius=1)

    if ndi is not None and segmentation is not None:
        distance = cast(np.ndarray, ndi.distance_transform_edt(binary))
        local_max = distance == ndi.maximum_filter(distance, size=max(3, min_distance * 2 + 1))
        markers = ndi_label(local_max & binary)
        labels = segmentation.watershed(-distance, markers, mask=binary)
        return labels.astype(np.int32)

    return label_connected_components(binary)


def binary_dilate(mask: np.ndarray, radius: int = 1) -> np.ndarray:
    padded = np.pad(mask.astype(bool), radius, mode="constant", constant_values=False)
    out = np.zeros_like(mask, dtype=bool)
    size = 2 * radius + 1
    for dy in range(size):
        for dx in range(size):
            out |= padded[dy : dy + mask.shape[0], dx : dx + mask.shape[1]]
    return out


def binary_erode(mask: np.ndarray, radius: int = 1) -> np.ndarray:
    padded = np.pad(mask.astype(bool), radius, mode="constant", constant_values=False)
    out = np.ones_like(mask, dtype=bool)
    size = 2 * radius + 1
    for dy in range(size):
        for dx in range(size):
            out &= padded[dy : dy + mask.shape[0], dx : dx + mask.shape[1]]
    return out


def binary_open(mask: np.ndarray, radius: int = 1) -> np.ndarray:
    return binary_dilate(binary_erode(mask, radius=radius), radius=radius)


def binary_close(mask: np.ndarray, radius: int = 1) -> np.ndarray:
    return binary_erode(binary_dilate(mask, radius=radius), radius=radius)


def fill_holes(mask: np.ndarray) -> np.ndarray:
    if ndi is not None:
      filled = cast(np.ndarray, ndi.binary_fill_holes(mask))
      return filled.astype(bool)

    mask = mask.astype(bool)
    background = ~mask
    h, w = mask.shape
    visited = np.zeros_like(mask, dtype=bool)
    stack: List[Tuple[int, int]] = []

    for x in range(w):
        if background[0, x]:
            stack.append((0, x))
        if background[h - 1, x]:
            stack.append((h - 1, x))
    for y in range(h):
        if background[y, 0]:
            stack.append((y, 0))
        if background[y, w - 1]:
            stack.append((y, w - 1))

    while stack:
        y, x = stack.pop()
        if y < 0 or y >= h or x < 0 or x >= w:
            continue
        if visited[y, x] or not background[y, x]:
            continue
        visited[y, x] = True
        stack.extend([(y - 1, x), (y + 1, x), (y, x - 1), (y, x + 1)])

    holes = background & ~visited
    return mask | holes

# Label each connected foreground object
# as an independent cell instance.

def label_connected_components(mask: np.ndarray) -> np.ndarray:
    mask = mask.astype(bool)

    if ndi is not None:
     labels = ndi_label(mask)
     return labels.astype(np.int32)

    if measure is not None:
      labels = cast(np.ndarray, measure.label(mask, connectivity=1))
      return labels.astype(np.int32)

    h, w = mask.shape
    labels = np.zeros((h, w), dtype=np.int32)
    current = 0
    ys, xs = np.nonzero(mask)

    for start_y, start_x in zip(ys, xs):
        if labels[start_y, start_x] != 0:
            continue

        current += 1
        stack = [(int(start_y), int(start_x))]
        labels[start_y, start_x] = current

        while stack:
            y, x = stack.pop()
            for ny, nx in ((y - 1, x), (y + 1, x), (y, x - 1), (y, x + 1)):
                if 0 <= ny < h and 0 <= nx < w and mask[ny, nx] and labels[ny, nx] == 0:
                    labels[ny, nx] = current
                    stack.append((ny, nx))

    return labels

from typing import cast

def ndi_label(mask: np.ndarray) -> np.ndarray:
    if ndi is None:
        raise RuntimeError("scipy.ndimage unavailable")

    labels, _ = cast(tuple[np.ndarray, int], ndi.label(mask))
    return labels


def area_limits(sequence_id: str, config: Config) -> Tuple[int, int]:
    family = sequence_family(sequence_id)
    if family == "hela":
        return 15, 4000
    if family == "pancreas":
        return 12, 2500
    if family == "hsc":
        return 15, 120000
    if family == "musc":
        return 15, 120000
    return config.min_area_default, config.max_area_default

# Remove implausible detections based on
# dataset-specific cell size constraints.

def clean_instance_mask(labels: np.ndarray, sequence_id: str, config: Config) -> np.ndarray:
    min_area, max_area = area_limits(sequence_id, config)
    labels = labels.astype(np.int32)
    clean_binary = np.zeros_like(labels, dtype=bool)

    for label_id in range(1, int(labels.max()) + 1):
        region = labels == label_id
        area = int(region.sum())
        if min_area <= area <= max_area:
            clean_binary |= region

    clean_binary = fill_holes(clean_binary)
    clean_binary = binary_open(clean_binary, radius=1)
    clean_binary = binary_close(clean_binary, radius=1)
    return label_connected_components(clean_binary)


def perimeter_from_region(region: np.ndarray) -> float:
    eroded = binary_erode(region, radius=1)
    border = region & ~eroded
    return float(border.sum())

# Compute geometric and intensity features
# for every segmented cell.


def extract_detections(
    image: np.ndarray,
    labels: np.ndarray,
    sequence_id: str,
    frame_number: int,
    mask_path: Path,
) -> List[Detection]:
    detections: List[Detection] = []
    max_label = int(labels.max())

    for label_id in range(1, max_label + 1):
        ys, xs = np.nonzero(labels == label_id)
        if len(xs) == 0:
            continue

        area = int(len(xs))
        x1 = int(xs.min())
        y1 = int(ys.min())
        x2 = int(xs.max() + 1)
        y2 = int(ys.max() + 1)
        centroid_x = float(xs.mean())
        centroid_y = float(ys.mean())
        mean_intensity = float(image[ys, xs].mean())

        region = labels[y1:y2, x1:x2] == label_id
        perimeter = perimeter_from_region(region)
        circularity = 0.0
        if perimeter > 0:
            circularity = float(4.0 * math.pi * area / (perimeter * perimeter))

        detections.append(
            Detection(
                sequence_id=sequence_id,
                frame_number=frame_number,
                detection_id=len(detections) + 1,
                label_id=label_id,
                area=area,
                centroid_x=centroid_x,
                centroid_y=centroid_y,
                bbox_x1=x1,
                bbox_y1=y1,
                bbox_x2=x2,
                bbox_y2=y2,
                mean_intensity=mean_intensity,
                perimeter=perimeter,
                circularity=circularity,
                mask_path=str(mask_path),
            )
        )

    return detections


def bbox_iou(a: Tuple[int, int, int, int], b: Tuple[int, int, int, int]) -> float:
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    iw = max(0, ix2 - ix1)
    ih = max(0, iy2 - iy1)
    intersection = iw * ih
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - intersection
    if union <= 0:
        return 0.0
    return float(intersection / union)

# Compute matching cost between an existing track
# and a new detection using spatial and appearance cues.

def detection_cost(track: Track, detection: Detection, config: Config) -> float:
    pred_x, pred_y = track.predicted_centroid()
    dx = pred_x - detection.centroid_x
    dy = pred_y - detection.centroid_y
    distance = math.hypot(dx, dy)
    if distance > config.max_tracking_distance:
        return 1e6

    prev = track.last_detection
    distance_cost = min(distance / config.max_tracking_distance, 1.0)
    iou_cost = 1.0 - bbox_iou(prev.bbox, detection.bbox)
    area_cost = abs(prev.area - detection.area) / max(prev.area, detection.area, 1)
    intensity_cost = abs(prev.mean_intensity - detection.mean_intensity)

    return float(
        config.distance_weight * distance_cost
        + config.iou_weight * iou_cost
        + config.area_weight * area_cost
        + config.intensity_weight * intensity_cost
    )


def assign_detections(
    active_tracks: List[Track],
    detections: List[Detection],
    config: Config,
) -> List[Tuple[int, int, float]]:
    if not active_tracks or not detections:
        return []

    costs = np.zeros((len(active_tracks), len(detections)), dtype=np.float32)
    for i, track in enumerate(active_tracks):
        for j, detection in enumerate(detections):
            costs[i, j] = detection_cost(track, detection, config)

    assignments: List[Tuple[int, int, float]] = []
    if linear_sum_assignment is not None:
        rows, cols = linear_sum_assignment(costs)
        for row, col in zip(rows, cols):
            cost = float(costs[row, col])
            if cost <= config.match_cost_threshold:
                assignments.append((int(row), int(col), cost))
        return assignments

    candidates: List[Tuple[float, int, int]] = []
    for row in range(costs.shape[0]):
        for col in range(costs.shape[1]):
            if costs[row, col] <= config.match_cost_threshold:
                candidates.append((float(costs[row, col]), row, col))

    used_rows = set()
    used_cols = set()
    for cost, row, col in sorted(candidates):
        if row in used_rows or col in used_cols:
            continue
        used_rows.add(row)
        used_cols.add(col)
        assignments.append((row, col, cost))

    return assignments


def find_parent_id(
    finished_or_missing_tracks: Iterable[Track],
    detection: Detection,
    config: Config,
) -> int:

    best_parent = 0
    best_score = float("inf")

    for track in finished_or_missing_tracks:

        # Need at least 2 detections to estimate motion
        if len(track.detections) < 2:
            continue

        parent = track.last_detection

        frame_gap = detection.frame_number - parent.frame_number

        # Parent must come from an earlier frame
        if frame_gap <= 0:
            continue

        # Allow parents that disappeared for a few frames
        if frame_gap > config.max_missing_frames + 1:
            continue

        predicted_x, predicted_y = track.predicted_centroid()

        distance = math.hypot(
            predicted_x - detection.centroid_x,
            predicted_y - detection.centroid_y,
        )

        if distance > config.max_tracking_distance * 0.8:
            continue

        area_ratio = detection.area / max(parent.area, 1)

        if not (0.30 <= area_ratio <= 0.80):
            continue

        intensity_diff = abs(
            parent.mean_intensity -
            detection.mean_intensity
        )

        if intensity_diff > 0.35:
            continue

        circularity_diff = abs(
            parent.circularity -
            detection.circularity
        )

        bbox_overlap = bbox_iou(
            parent.bbox,
            detection.bbox,
        )

        velocity_penalty = 0.0

        if len(track.detections) >= 2:

            previous = track.detections[-2]

            vx = parent.centroid_x - previous.centroid_x
            vy = parent.centroid_y - previous.centroid_y

            expected_x = parent.centroid_x + vx
            expected_y = parent.centroid_y + vy

            velocity_penalty = math.hypot(
                expected_x - detection.centroid_x,
                expected_y - detection.centroid_y,
            )

        score = (
            distance * 0.35
            + abs(area_ratio - 0.5) * 35.0
            + intensity_diff * 15.0
            + circularity_diff * 10.0
            + (1.0 - bbox_overlap) * 12.0
            + velocity_penalty * 0.15
        )

        if score < best_score:
            best_score = score
            best_parent = track.track_id

    return best_parent


def track_sequence(
    sequence_id: str,
    frame_detections: Dict[int, List[Detection]],
    config: Config,
) -> List[Track]:
    active_tracks: List[Track] = []
    finished_tracks: List[Track] = []
    next_track_id = 1

    for frame_number in sorted(frame_detections):
        detections = frame_detections[frame_number]

        if not active_tracks:
            for detection in detections:
                track = Track(track_id=next_track_id)
                track.add_detection(detection)
                active_tracks.append(track)
                next_track_id += 1
            continue

        import copy

        parent_candidates = copy.deepcopy(active_tracks)
        assignments = assign_detections(active_tracks, detections, config)
        matched_tracks = {row for row, _, _ in assignments}
        matched_detections = {col for _, col, _ in assignments}

        for track_index, detection_index, _ in assignments:
            active_tracks[track_index].add_detection(detections[detection_index])

        newly_finished: List[Track] = []
        still_active: List[Track] = []
        for index, track in enumerate(active_tracks):
            if index in matched_tracks:
                still_active.append(track)
                continue

            track.missing_count += 1
            if track.missing_count > config.max_missing_frames:
                track.active = False
                newly_finished.append(track)
            else:
                still_active.append(track)

        missing_context = list(newly_finished)
        missing_context.extend(track for i, track in enumerate(active_tracks) if i not in matched_tracks)
        finished_tracks.extend(newly_finished)
        active_tracks = still_active

        for index, detection in enumerate(detections):
            if index in matched_detections:
                continue
        
            candidate_parents = [
    t
    for t in (parent_candidates + finished_tracks)
    if t.last_detection.frame_number < frame_number
]

            parent_id = find_parent_id(
    candidate_parents,
    detection,
    config,
)

            
            track = Track(track_id=next_track_id, parent_id=parent_id)
            track.add_detection(detection)
            active_tracks.append(track)
            next_track_id += 1

    for track in active_tracks:
        track.active = False
        finished_tracks.append(track)

    return finished_tracks


def save_mask(labels: np.ndarray, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    labels = labels.astype(np.uint16 if labels.max() > 255 else np.uint8)
    Image.fromarray(labels).save(path)


def save_visualization(
    image: np.ndarray,
    labels: np.ndarray,
    detections: Sequence[Detection],
    path: Path,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    base = Image.fromarray(float_to_uint8(image), mode="L")
    rgb = ImageOps.colorize(base, black="black", white="white").convert("RGB")
    draw = ImageDraw.Draw(rgb)

    for detection in detections:
        color = color_for_id(detection.track_id or detection.detection_id)
        draw.rectangle(detection.bbox, outline=color, width=2)
        label = str(detection.track_id or "?")
        draw.text((detection.bbox_x1, max(0, detection.bbox_y1 - 12)), label, fill=color)

    # Draw light mask boundaries last enough to be visible but not hide image detail.
    boundary = labels > 0
    eroded = binary_erode(boundary, radius=1)
    border_y, border_x = np.nonzero(boundary & ~eroded)
    from typing import Any, cast

    pixels = cast(Any, rgb.load())
    for y, x in zip(border_y, border_x):
        pixels[int(x), int(y)] = (255, 220, 0)

    rgb.save(path)


def color_for_id(track_id: int) -> Tuple[int, int, int]:
    rng = random.Random(track_id * 7919)
    return rng.randint(80, 255), rng.randint(80, 255), rng.randint(80, 255)


def tracks_to_dataframe(tracks: Sequence[Track]) -> pd.DataFrame:
    rows = []
    for track in tracks:
        for detection in track.detections:
            rows.append(
                {
                    "sequence_id": detection.sequence_id,
                    "frame_number": detection.frame_number,
                    "track_id": detection.track_id,
                    "parent_id": detection.parent_id,
                    "centroid_x": detection.centroid_x,
                    "centroid_y": detection.centroid_y,
                    "area": detection.area,
                    "bbox_x1": detection.bbox_x1,
                    "bbox_y1": detection.bbox_y1,
                    "bbox_x2": detection.bbox_x2,
                    "bbox_y2": detection.bbox_y2,
                    "mean_intensity": detection.mean_intensity,
                    "perimeter": detection.perimeter,
                    "circularity": detection.circularity,
                    "mask_path": detection.mask_path,
                }
            )
    return pd.DataFrame(rows)

# Complete processing pipeline for a single sequence:
# preprocessing -> segmentation -> detection -> tracking -> output.

def run_sequence(sequence: SequenceRecord, config: Config, model=None, device=None) -> pd.DataFrame:
    print(f"Processing {sequence.sequence_id}: {len(sequence.frame_paths)} frames")

    frame_detections: Dict[int, List[Detection]] = {}
    frame_images: Dict[int, np.ndarray] = {}
    frame_labels: Dict[int, np.ndarray] = {}
    frame_stems: Dict[int, str] = {}

    for path, frame_number in zip(sequence.frame_paths, sequence.frame_numbers):
        image = load_frame(path)
        processed = preprocess_frame(image)

        if config.segmentation_method == "model":
            if model is None or device is None:
                raise RuntimeError("Model segmentation requested, but no model was loaded.")
            raw_labels = segment_cells_with_model(processed, model, device, config)
        elif config.segmentation_method == "auto" and model is not None and device is not None:
            raw_labels = segment_cells_with_model(processed, model, device, config)
        else:
            raw_labels = segment_cells(processed, sequence.sequence_id)

        labels = clean_instance_mask(raw_labels, sequence.sequence_id, config)

        frame_stems[frame_number] = path.stem
        mask_path = config.masks_dir / sequence.sequence_id / f"{path.stem}_mask.png"
        if config.save_masks:
            save_mask(labels, mask_path)

        detections = extract_detections(
            image=image,
            labels=labels,
            sequence_id=sequence.sequence_id,
            frame_number=frame_number,
            mask_path=mask_path,
        )
        frame_detections[frame_number] = detections
        frame_images[frame_number] = image
        frame_labels[frame_number] = labels

    tracks = track_sequence(sequence.sequence_id, frame_detections, config)
    df = tracks_to_dataframe(tracks)

    sequence_csv = config.tracks_dir / f"{sequence.sequence_id}_tracks.csv"
    df.to_csv(sequence_csv, index=False)

    if config.save_visualizations:
        visual_frame_numbers = visualization_frame_numbers(sorted(frame_detections), config.visualize_every)
        by_frame: Dict[int, List[Detection]] = {}
        for track in tracks:
            for detection in track.detections:
                by_frame.setdefault(detection.frame_number, []).append(detection)

        for frame_number in visual_frame_numbers:
            output_path = config.visualizations_dir / sequence.sequence_id / f"{frame_stems[frame_number]}_overlay.png"
            save_visualization(
                frame_images[frame_number],
                frame_labels[frame_number],
                by_frame.get(frame_number, []),
                output_path,
            )

    print(f"Finished {sequence.sequence_id}: {len(tracks)} tracks, {len(df)} detections")
    return df


def visualization_frame_numbers(frame_numbers: Sequence[int], every: int) -> List[int]:
    if not frame_numbers:
        return []
    if every <= 0:
        return [frame_numbers[0], frame_numbers[-1]]

    selected = [number for idx, number in enumerate(frame_numbers) if idx % every == 0]
    if frame_numbers[-1] not in selected:
        selected.append(frame_numbers[-1])
    return selected


def save_summary(sequences: Sequence[SequenceRecord], all_tracks: pd.DataFrame, config: Config) -> None:
    summary_rows = []
    for sequence in sequences:
        seq_df = all_tracks[all_tracks["sequence_id"] == sequence.sequence_id]
        summary_rows.append(
            {
                "sequence_id": sequence.sequence_id,
                "frames": len(sequence.frame_paths),
                "width": sequence.image_size[0],
                "height": sequence.image_size[1],
                "bit_depth": sequence.bit_depth,
                "detections": int(len(seq_df)),
                "tracks": int(seq_df["track_id"].nunique()) if not seq_df.empty else 0,
            }
        )

    summary = pd.DataFrame(summary_rows)
    summary.to_csv(config.logs_dir / "run_summary.csv", index=False)


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="BioHub cell tracking pipeline")
    parser.add_argument("--mode", choices=["train", "infer", "train-infer"], default="infer", help="Run training, inference/tracking, or both")
    parser.add_argument("--dataset", default="dataset-smaller", help="Dataset folder")
    parser.add_argument("--mask-dir", default=None, help="Ground-truth mask folder for supervised training")
    parser.add_argument("--output", default="outputs", help="Output folder")
    parser.add_argument("--checkpoint", default=None, help="Checkpoint path for model inference")
    parser.add_argument("--segmentation", choices=["auto", "model", "classical"], default="auto", help="Segmentation source during inference")
    parser.add_argument("--image-size", type=int, default=512, help="Training/inference model image size")
    parser.add_argument("--batch-size", type=int, default=4, help="Training batch size")
    parser.add_argument("--epochs", type=int, default=50, help="Training epochs")
    parser.add_argument("--learning-rate", type=float, default=1e-3, help="AdamW learning rate")
    parser.add_argument("--weight-decay", type=float, default=1e-4, help="AdamW weight decay")
    parser.add_argument("--val-split", type=float, default=0.2, help="Fraction of labeled pairs used for validation")
    parser.add_argument("--num-workers", type=int, default=0, help="DataLoader worker count")
    parser.add_argument("--threshold", type=float, default=0.5, help="Segmentation probability threshold")
    parser.add_argument("--sequence", default=None, help="Process only one sequence")
    parser.add_argument("--limit-frames", type=int, default=None, help="Limit frames per sequence")
    parser.add_argument("--visualize-every", type=int, default=25, help="Overlay every N frames")
    parser.add_argument("--no-masks", action="store_true", help="Do not save masks")
    parser.add_argument("--no-visuals", action="store_true", help="Do not save overlay images")
    return parser.parse_known_args()[0]

# Entry point for training and inference.


def main() -> None:
    args = parse_args()
    config = Config(
        dataset_dir=Path(args.dataset),
        output_dir=Path(args.output),
        train_mask_dir=Path(args.mask_dir) if args.mask_dir else None,
        checkpoint_path=Path(args.checkpoint) if args.checkpoint else None,
        image_size=args.image_size,
        batch_size=args.batch_size,
        epochs=args.epochs,
        learning_rate=args.learning_rate,
        weight_decay=args.weight_decay,
        val_split=args.val_split,
        num_workers=args.num_workers,
        confidence_threshold=args.threshold,
        segmentation_method=args.segmentation,
        limit_frames=args.limit_frames,
        only_sequence=args.sequence,
        visualize_every=args.visualize_every,
        save_masks=not args.no_masks,
        save_visualizations=not args.no_visuals,
    )

    set_seed(config.seed)
    create_output_folders(config)

    trained_checkpoint = None
    if args.mode in {"train", "train-infer"}:
        trained_checkpoint = train_model(config)
        if args.mode == "train":
            return
        config.checkpoint_path = trained_checkpoint

    sequences = discover_sequences(config)
    if not sequences:
        raise RuntimeError(f"No PNG sequences found under {config.dataset_dir}")

    model = None
    device = None
    if config.segmentation_method in {"auto", "model"}:
        checkpoint_candidate = config.checkpoint_path or (config.checkpoint_dir / "best_unet.pt")
        if checkpoint_candidate.exists():
            model, device = load_checkpoint_model(config)
            print(f"Using trained segmentation model: {checkpoint_candidate}")
        elif config.segmentation_method == "model":
            raise FileNotFoundError(
                f"--segmentation model requires a checkpoint, but none was found at {checkpoint_candidate}"
            )
        else:
            print("No model checkpoint found; falling back to classical segmentation.")

    all_frames = []
    for sequence in sequences:
        all_frames.append(run_sequence(sequence, config, model=model, device=device))

    all_tracks = pd.concat(all_frames, ignore_index=True) if all_frames else pd.DataFrame()
    all_tracks.to_csv(config.tracks_dir / "all_tracks.csv", index=False)
    save_summary(sequences, all_tracks, config)

    print(f"Done. Results saved under {config.output_dir}")
    

if __name__ == "__main__":
    main()


No model checkpoint found; falling back to classical segmentation.
Processing HeLa-01: 92 frames
Finished HeLa-01: 226 tracks, 6629 detections
Processing HeLa-02: 92 frames
Finished HeLa-02: 632 tracks, 18632 detections
Processing HSC-01: 1764 frames
Finished HSC-01: 2298 tracks, 35019 detections
Processing HSC-02: 1764 frames
Finished HSC-02: 3963 tracks, 77990 detections
Processing MuSC-01: 1376 frames
Finished MuSC-01: 5203 tracks, 74603 detections
Processing MuSC-02: 1376 frames
Finished MuSC-02: 8830 tracks, 92391 detections
Processing pancreas-01: 300 frames
Finished pancreas-01: 1441 tracks, 75881 detections
Processing pancreas-02: 300 frames
Finished pancreas-02: 1278 tracks, 62232 detections
Done. Results saved under outputs
